### Search am Beispiel Maze


In [82]:
from collections import deque
from heapq import heappop, heappush

def bfs(startstate):
    ''' 
    returns: Tupel (prev, state) 
        prev: dictionary mit den Vorgängern der untersuchten Spielstellungen,            
        state: Spielstellung, die den goaltest besteht
        wenn Ziel nicht gefunden: None, None
    '''   
    frontier =  deque([startstate])
    prev = {startstate:None}
    while frontier:
        state = frontier.popleft()  
        if goaltest(state):
            return prev, state, frontier
        for v in nextstates(state):
            if v not in prev:
                frontier.append(v)
                prev[v] = state
    return None, None, None

def h(state):
    ''' Manhatten-Distanz '''
    x1, y1 = state
    x2, y2 = goalstate
    return abs(x1-x2) + abs(y1-y2)

def greedy(startstate):
    frontier =[(h(startstate),startstate)]  
    prev = {startstate:None}
    while frontier:
        _ ,state = heappop(frontier)  
        if goaltest(state):
            return prev,state, frontier
        for v in nextstates(state):
            if v not in prev:
                heappush(frontier,(h(v),v))
                prev[v] = state
    return None, None

def astar(startstate):
    frontier =[(h(startstate),startstate)]  
    prev = {startstate:None}
    g = {startstate:0}          # bisher angefallenen Kosten: die Anzahl Züge
    while frontier:
        _ ,state = heappop(frontier)   
        if goaltest(state):
            return prev,state, frontier
        for v in nextstates(state):
            gg = g[state] + 1
            if v not in prev or gg < g[v]:   # v noch nicht explored und nicht in frontier oder die Rückwärtskosten sind geringer
                g[v] = gg                    # als die bisherigen Rückwärtskosten von v.
                heappush(frontier,(g[v]+h(v),v))
                prev[v] = state
    return None, None

def reconstructPath(prev,goalstate):
    state = goalstate
    path = []
    while state is not None:
        path.append(state)
        state = prev[state]
    path.reverse()
    return path

def goaltest(state):
    return grid[state[0]][state[1]] == 'E'

def nextstates(state):
    x, y = state
    tmp = []
    for xd, yd in [(0,1),(0,-1),(1,0),(-1,0)]:
        xn, yn = x+xd, y+yd
        if grid[xn][yn] != 'X':
            tmp.append((xn,yn))
    return tmp

def getStartAndGoal(grid):
    for x in range(len(grid)):
        for y in range(len(grid[x])):
            if grid[x][y] == 'S':
                startstate = (x, y)
            if grid[x][y] == 'E':
                goalstate = (x,y)
    return startstate, goalstate
    


def showGrid(path, prev, frontier):
    grid = [list(s) for s in lines]
    for x,y in prev:
        grid[x][y]='.'
    for x,y in path:
        grid[x][y] = 'o'
    for tup in frontier:
        if isinstance(tup[1],int)   == 1:  # bfs
            x, y = tup
        else:
            x, y = tup[1]     # greedy, astart

        grid[x][y] = '~'
    x,y = path[0]
    grid[x][y] = 'S'
    x,y = path[-1]
    grid[x][y] = 'E'
    for row in grid:
        print(''.join(row))
 
    print("explored = {}, weglänge = {}".format(len(prev)-len(frontier), len(path)-1))
    print("gefundener Weg: 'o', explored: '.', frontier: '~'")
    

In [100]:
%%writefile maze.txt
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X                                                                 X
X                                                                 X
X                                                                 X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       XXXXXXXXXXXXX             X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       X                         X  
X                                       X                         X
X        XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX   E                     X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                            S          X                         X
X                                       X                         X
X        XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX                         X                         
X                                                                 X
X                                                                 X
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Overwriting maze.txt


In [101]:
f = open("maze.txt")       
lines = f.read().splitlines()
f.close()

In [102]:
grid = [list(s) for s in lines]
startstate, goalstate = getStartAndGoal(grid)

prev, state, frontier = bfs(startstate)
if prev is None:
    print('Keine Lösung gefunden')
else:
    path = reconstructPath(prev, state)
    print(f'Bfs Lösung in {len(path)-1} Schritten')

 
showGrid(path, prev, frontier)
  

Bfs Lösung in 67 Schritten
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X..................................~                              X
X...................................~                             X
X....................................~                            X
X.....................................~ X                         X
X......................................~X                         X
X.......................................X                         X
X.......................................X                         X
X.......................................XXXXXXXXXXXXX             X
X.......................................X                         X
X.......................................X                         X
X.......................................X                         X
X.......................................X                         X
X.......................................X~                        X
X....................

In [103]:
grid = [list(s) for s in lines]
prev, state, frontier = greedy(startstate)
if prev is None:
    print('Keine Lösung gefunden')
else:
    path = reconstructPath(prev, state)
    print(f'Greedy Lösung in {len(path)-1} Schritten')

showGrid(path, prev, frontier)

Greedy Lösung in 105 Schritten
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X                                       ~.......~                 X
X                                      ~.........~                X
X                                     ~oooooo.....~               X
X                                    ~.oX...o......~              X
X                                   ~..oX...o.......~             X
X                                  ~...oX...o........~            X
X                                 ~....oX...oooooooooo~           X
X                                 ~....oXXXXXXXXXXXXXo~           X
X                                ~.....oX  ~oooooooooo~           X
X                               ~......oX  ~o~~~~~~~~~            X
X                              ~.......oX  ~o~                    X
X                             ~........oX  ~o~                    X
X                            ~.........oX  ~o~                    X
X       ~~~~~~~~~

In [104]:
grid = [list(s) for s in lines]
prev, state, frontier = astar(startstate)
if prev is None:
    print('Keine Lösung gefunden')
else:
    path = reconstructPath(prev, state)
    print(f'Astar Lösung in {len(path)-1} Schritten')

showGrid(path, prev, frontier)

Astar Lösung in 67 Schritten
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X                                                                 X
X                                                                 X
X                                                                 X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X                                       XXXXXXXXXXXXX             X
X                                       X                         X
X                                       X                         X
X                                       X                         X
X       ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~X                         X
X      ~................................X                         X
X     ~............

In [11]:
f = open("data/maze02.txt")       
lines = f.read().splitlines()
grid = [list(s) for s in lines]
startstate = getStart()

prev, goalstate, frontier = bfs(startstate)     # bfs, dfs 
path = reconstructPath(prev,goalstate)

showResult()
    
 

FileNotFoundError: [Errno 2] No such file or directory: 'data/maze02.txt'


Codingame
- [11-Puzzle](https://www.codingame.com/training/hard/11-puzzle)
- [Labyrinth](https://www.codingame.com/training/hard/the-labyrinth)